# 02 — Feature Engineering

EDA çıktılarına dayanarak modele hazır feature seti oluşturuyoruz.

**Giriş:** `load_season_agg()` → her (lokasyon × bitki × yıl × WAV) için tek satır  
**Çıkış:** `data/processed/features.parquet` — train/test split dahil

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.loader import load_season_agg

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

## 1. Veriyi yükle

In [2]:
print('Sezon aggregate yükleniyor...')
agg = load_season_agg()
print(f'Toplam satır: {len(agg):,}')
agg.head()

Sezon aggregate yükleniyor...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Toplam satır: 53,184


,latitude,longitude,elevation,crop_name,variety_name,year,wav_scenario,WAV,harvest_twso,sim_success,mean_temp,total_precip,mean_humidity,mean_rftra,min_rftra,max_lai,max_tagp,max_dvs,season_days
0,38.0,42.5,1607.0,barley,Spring_barley_301,2014,dry,10,2985.219264,1,10.976361,720.1,57.078439,0.510998,0.0,6.250648,9917.655012,2.00,366
1,38.0,42.5,1607.0,barley,Spring_barley_301,2014,wet,100,4170.508313,1,10.976361,720.1,57.078439,0.552700,0.0,6.301078,11332.161102,2.00,366
2,40.0,29.0,1125.0,barley,Spring_barley_301,2014,dry,10,9149.285820,1,10.355740,1170.0,76.574009,0.872129,0.0,6.193121,16686.455022,2.00,366
3,40.0,29.0,1125.0,barley,Spring_barley_301,2014,wet,100,9149.285820,1,10.355740,1170.0,76.574009,0.872129,0.0,6.193121,16686.455022,2.00,366
4,36.0,42.5,218.0,cassava,Cassava_VanHeemst_1988,2014,normal,50,0.928732,1,21.791926,318.7,47.483665,0.282382,0.0,2.906455,2170.549756,1.58,366


## 2. Sadece başarılı hasatlar

In [3]:
df = agg[agg['sim_success'] == 1].copy()
print(f'Başarılı hasat: {len(df):,} ({len(df)/len(agg):.1%})')
print(f'Bitki sayısı: {df["crop_name"].nunique()}')
print(f'Yıl aralığı: {df["year"].min()} – {df["year"].max()}')

Başarılı hasat: 41,423 (77.9%)
Bitki sayısı: 22
Yıl aralığı: 2014 – 2024


## 3. Feature seçimi

EDA'dan öğrendiklerimiz:
- `mean_rftra` (r=0.52) ve `max_tagp` (r=0.50) en güçlü sinyaller
- `min_rftra` anlamsız (EDA'da düz)
- `mean_temp` tek başına zayıf ama diğerleriyle birlikte tutuyoruz
- `latitude`, `longitude` uzamsal örüntü taşıyor
- `year` yıllık iklim varyasyonunu yakalar

In [4]:
FEATURE_COLS = [
    # Hava
    'mean_temp', 'total_precip', 'mean_humidity',
    # Su stresi
    'mean_rftra',
    # Büyüme
    'max_lai', 'max_tagp', 'max_dvs', 'season_days',
    # Konum
    'latitude', 'longitude', 'elevation',
    # Zaman
    'year',
    # Senaryo
    'WAV',
    # Bitki (target encoded aşağıda)
    'crop_name',
]

TARGET = 'harvest_twso'

df = df[FEATURE_COLS + [TARGET]].reset_index(drop=True)
print(df.shape)
df.isnull().sum()

(41423, 15)


mean_temp        0
total_precip     0
mean_humidity    0
mean_rftra       0
max_lai          0
max_tagp         0
max_dvs          0
season_days      0
latitude         0
longitude        0
elevation        0
year             0
WAV              0
crop_name        0
harvest_twso     0
dtype: int64

## 4. Train / Test split (yıl bazında)

Son 2 yılı test olarak ayırıyoruz — random split yaparsak veri sızıntısı olur (aynı lokasyon farklı yıllarda hem train hem test'te çıkar, bu kabul edilebilir ama zaman serisinde gerçekçi değil).

In [5]:
test_years = sorted(df['year'].unique())[-2:]
print(f'Test yılları: {test_years}')

train = df[~df['year'].isin(test_years)].copy()
test  = df[ df['year'].isin(test_years)].copy()

print(f'Train: {len(train):,} satır  |  Test: {len(test):,} satır')

Test yılları: [np.int64(2023), np.int64(2024)]
Train: 34,013 satır  |  Test: 7,410 satır


## 5. Target Encoding (crop_name)

Sadece train üzerinden hesaplanır, test'e uygulanır — leakage önlemi.

In [ ]:
import json

crop_mean = train.groupby('crop_name')[TARGET].mean().rename('crop_te')
print('Bitki başına ortalama verim (target encoding):')
print(crop_mean.sort_values().to_string())

train = train.join(crop_mean, on='crop_name').drop(columns='crop_name')
test  = test.join(crop_mean, on='crop_name').drop(columns='crop_name')

global_mean = train[TARGET].mean()
test['crop_te'] = test['crop_te'].fillna(global_mean)

# Mapping'i kaydet — diğer notebook'lar bu dosyayı yükleyecek
crop_te_export = crop_mean.to_dict()
crop_te_export['__global_mean__'] = global_mean
with open(PROCESSED / 'crop_te_map.json', 'w') as f:
    json.dump(crop_te_export, f, indent=2)
print('crop_te_map.json kaydedildi.')

In [7]:
feature_cols = [c for c in train.columns if c != TARGET]
print(f'Feature sayısı: {len(feature_cols)}')
print(feature_cols)

train[feature_cols].describe().T

Feature sayısı: 14
['mean_temp', 'total_precip', 'mean_humidity', 'mean_rftra', 'max_lai', 'max_tagp', 'max_dvs', 'season_days', 'latitude', 'longitude', 'elevation', 'year', 'WAV', 'crop_te']


,count,mean,std,min,25%,50%,75%,max
mean_temp,34013.0,13.098242,4.274800,0.147467,10.355740,12.694402,15.355675,22.790839
total_precip,34013.0,692.453268,383.709378,102.600000,429.300000,618.500000,851.300000,2705.100000
mean_humidity,34013.0,63.312459,9.603776,34.925166,56.841021,63.634269,70.072151,84.902049
mean_rftra,34013.0,0.508713,0.193534,0.000000,0.379317,0.492039,0.640268,0.956522
max_lai,34013.0,4.091047,2.966075,0.000920,1.678086,3.984280,5.893164,15.625269
max_tagp,34013.0,6200.975925,5274.037061,0.849488,2324.783580,5087.698095,9064.040048,32562.739931
max_dvs,34013.0,1.531780,0.624628,-0.077570,1.319356,1.806516,2.000000,2.000000
season_days,34013.0,366.220680,0.414711,366.000000,366.000000,366.000000,366.000000,367.000000
latitude,34013.0,38.994532,1.777201,36.000000,38.000000,39.000000,40.000000,42.000000
longitude,34013.0,35.533178,5.453048,26.000000,30.500000,35.000000,41.000000,44.000000


## 8. Kaydet

In [8]:
train.to_parquet(PROCESSED / 'train.parquet', index=False)
test.to_parquet(PROCESSED / 'test.parquet', index=False)

print(f'Train kaydedildi: {PROCESSED}/train.parquet  ({len(train):,} satır)')
print(f'Test  kaydedildi: {PROCESSED}/test.parquet   ({len(test):,} satır)')

Train kaydedildi: ../data/processed/train.parquet  (34,013 satır)
Test  kaydedildi: ../data/processed/test.parquet   (7,410 satır)
